# GQA — upload verifier và submission để chấm

1. Cài kickmix qua pip.
2. Upload `gqa_verifiers.zip` (code chấm gốc và dev harness).
3. Chọn bài/chế độ.
4. Upload submission rồi chạy verify.
5. Tải results.

Không nhúng source hay circuit bằng base64. Ghi full PASS và dev_5min PASS vào SCORES.md, phân biệt rõ chế độ.


In [ ]:
import sys
if sys.version_info[:2] != (3, 13):
    raise RuntimeError('Select a Python 3.13 Colab runtime before installing baseline dependencies')
%pip -q install kickmix==1.0.0 numpy==2.4.6 scipy==1.16.3


In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, io, sys, os, json, subprocess, datetime, uuid, hashlib
import importlib.metadata

ROOT = Path('/content/GQA_verify')
ROOT.mkdir(exist_ok=True)
print('Upload gqa_verifiers.zip từ artifacts/colab/ của project')
uploaded = files.upload()
assert len(uploaded) == 1, 'Chọn đúng một ZIP verifier'
name, content = next(iter(uploaded.items()))
assert name.endswith('.zip')
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    for entry in archive.infolist():
        name = entry.filename
        assert chr(92) not in name and not name.startswith('/')
        assert '..' not in Path(name).parts
        assert (name.startswith('verifiers/') and name.endswith('.py')) or name == 'src/common/dev_verify.py', name
    archive.extractall(ROOT)
del uploaded, content
sys.path.insert(0, str(ROOT/'verifiers'))
import kickmix as km
print('kickmix:', importlib.metadata.version('kickmix'), km.__file__)
print('Verifier:', ROOT/'verifiers')


## Chọn bài và chế độ
`dev_5min`: mục tiêu khoảng 5 phút, timeout 300 giây tính từ lúc mở verifier (không tính upload/pip). Không bảo đảm tương đương full. Giữ seed/số mẫu giống nhau khi so sánh. TIMEOUT không phải lỗi mạch và không có điểm hợp lệ.
Invsqrt: 4.096 x trải toàn miền, gồm biên chuẩn hóa/đa thức, lặp 2 lần. DQI: 4.096 shot, giữ nguyên bài. DFTHC: mọi địa chỉ, 4 shot/địa chỉ, giữ đầy đủ chỉ số Hamiltonian.
Full invsqrt có 200.540.928 shot. DQI dùng một worker.


In [ ]:
TRACK='invsqrt' #@param ['invsqrt','dqi','dfthcsos']
MODE='dev_5min' #@param ['smoke','dev_5min','full']
EXPERIMENT_ID='INV-BASELINE' #@param {type:'string'}
OWNER='team-member' #@param {type:'string'}

# Giữ cùng cấu hình/seed giữa các phương án khi so sánh.
DEV_TIMEOUT_SECONDS=300 #@param {type:'integer'}
DEV_SEED=20261106 #@param {type:'integer'}
DEV_INVSQRT_SAMPLES=4096 #@param {type:'integer'}
DEV_INVSQRT_REPEATS=2 #@param {type:'integer'}
DEV_DQI_SHOTS=4096 #@param {type:'integer'}
DEV_DFTHC_REPEATS=4 #@param {type:'integer'}


## Upload submission — bắt buộc đúng một file
- Task 1 (invsqrt) / Task 2 (DQI): đúng **1 file `.kmb`**.
- Task 3 (DFTHC): đúng **1 file `.zip`** chứa `submission.npz` và toàn bộ `angle_k_*.kmb` liên tục từ 0, ở gốc ZIP. Không nhận các file rời.
- Sai số lượng/định dạng/cấu trúc ZIP: notebook báo lỗi; bỏ bớt hoặc sửa file và chạy lại cell upload. Cell verify sẽ bị chặn, kể cả khi trước đó từng upload thành công.
- Task 3 vẫn nạp riêng `dfthcsos_reference.zip` ở cell tiếp theo nếu runtime chưa có dữ liệu gốc.

Tên config lấy từ tên file `.kmb` hoặc ZIP; ký tự ngoài chữ/số/gạch ngang/gạch dưới đổi thành `_`.


In [ ]:
import re
# Vô hiệu submission cũ ngay khi thử upload lại, kể cả lần mới báo lỗi.
SUBMISSION_READY=False
UPLOAD_ARTIFACT=None
SUBMISSION_TRACK=None
submission=None
CONFIG_NAME=None
config_source_name=None
uploaded=files.upload()
expected_ext='.zip' if TRACK=='dfthcsos' else '.kmb'
if len(uploaded)!=1:
    raise ValueError(f'Chỉ được upload đúng 1 file {expected_ext} cho bài {TRACK}; đã nhận {len(uploaded)} file. Hãy bỏ/xóa bớt file trong lựa chọn và chạy lại cell upload.')
upload_name,content=next(iter(uploaded.items()))
name=Path(upload_name).name
if Path(name).suffix!=expected_ext:
    raise ValueError(f'Bài {TRACK} chỉ nhận đúng 1 file {expected_ext}. Hãy bỏ file sai định dạng và upload lại; không chạy verify.')

if TRACK=='dfthcsos':
    # Kiểm tra toàn bộ cấu trúc ZIP trước khi ghi file vào thư mục submission.
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        names=archive.namelist()
        if len(names)!=len(set(names)):
            raise ValueError('ZIP có tên file trùng nhau; hãy đóng gói lại.')
        if any('/' in item or chr(92) in item or not (item=='submission.npz' or re.fullmatch(r'angle_k_(0|[1-9][0-9]*)\.kmb',item)) for item in names):
            raise ValueError('ZIP chỉ được chứa submission.npz và angle_k_0.kmb, angle_k_1.kmb, ... ở thư mục gốc, không chứa thư mục con hoặc file khác.')
        indices=sorted(int(item[8:-4]) for item in names if item.startswith('angle_k_'))
        if 'submission.npz' not in names or not indices or indices!=list(range(len(indices))):
            raise ValueError('ZIP thiếu submission.npz hoặc các angle_k_*.kmb không liên tục từ 0. Hãy sửa ZIP và upload lại.')
        SUB=ROOT/'uploads'/uuid.uuid4().hex
        SUB.mkdir(parents=True,exist_ok=False)
        for item in names: (SUB/item).write_bytes(archive.read(item))
    submission=SUB
else:
    SUB=ROOT/'uploads'/uuid.uuid4().hex
    SUB.mkdir(parents=True,exist_ok=False)
    submission=SUB/name
    submission.write_bytes(content)
UPLOAD_ARTIFACT={'name':name,'sha256':hashlib.sha256(content).hexdigest(),'size_bytes':len(content)}
config_source_name=Path(name).stem
CONFIG_NAME=re.sub(r'[^A-Za-z0-9_-]+', '_', config_source_name).strip('_-') or 'default'
SUBMISSION_TRACK=TRACK
SUBMISSION_READY=True
# Không giữ thêm bản nội dung upload trong RAM khi chạy verifier.
del uploaded,content
print('Submission hợp lệ:',submission)
print('Config:',CONFIG_NAME)


In [ ]:
if TRACK=='dfthcsos':
    target=ROOT/'verifiers/dfthcsos'
    names=['FeMoCo76o113e.fcidump','FeMoCo76o113e.fixed_shifts.npz']
    if not all((target/n).is_file() for n in names):
        print('Upload dfthcsos_reference.zip (verifier tự kiểm tra checksum dữ liệu)')
        uploaded=files.upload()
        assert len(uploaded)==1
        with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
            assert set(z.namelist())==set(names)
            for name in names: (target/name).write_bytes(z.read(name))
        del uploaded


## Verify
Cell này chạy đến khi verifier kết thúc. Kết quả JSON và log được lưu cả khi FAIL. Đừng chạy thêm lượt khác khi cell còn bận.

In [ ]:
if not globals().get('SUBMISSION_READY',False) or globals().get('SUBMISSION_TRACK')!=TRACK:
    raise RuntimeError('Submission chưa hợp lệ hoặc đã đổi bài. Hãy chạy lại cell upload: task 1–2 đúng 1 .kmb, task 3 đúng 1 ZIP. Không chạy verifier.')
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*',EXPERIMENT_ID)
run_id=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-colab-'+MODE+'-'+uuid.uuid4().hex[:6]
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*',CONFIG_NAME), 'Tên config không hợp lệ'
OUT=ROOT/'results'/TRACK/EXPERIMENT_ID/CONFIG_NAME/MODE/run_id; OUT.mkdir(parents=True,exist_ok=False)
# Build command explicitly to keep the official verifier and defaults intact.
verifier=ROOT/'verifiers'/TRACK/('verify.py' if TRACK=='dfthcsos' else f'verify_submission_{TRACK}.py')
cmd=[sys.executable,str(verifier),str(submission)]
if TRACK in ['invsqrt','dqi']: cmd+=['--max-workers','1']
if MODE=='smoke':
    if TRACK=='invsqrt': cmd+=['--max-x','4096','--shots-per-x','1']
    elif TRACK=='dqi': cmd+=['--num-shots','1024']
    else: cmd+=['--shots-per-address','1']
if MODE=='dev_5min':
    assert DEV_TIMEOUT_SECONDS > 0
    cmd=[sys.executable,str(ROOT/'src/common/dev_verify.py'),'--track',TRACK,'--submission',str(submission),
         '--seed',str(DEV_SEED),'--invsqrt-samples',str(DEV_INVSQRT_SAMPLES),
         '--invsqrt-repeats',str(DEV_INVSQRT_REPEATS),'--dqi-shots',str(DEV_DQI_SHOTS),
         '--dfthc-repeats',str(DEV_DFTHC_REPEATS)]
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
meta={'experiment_id':EXPERIMENT_ID,'config_name':CONFIG_NAME,'config_source_name':config_source_name,'run_id':run_id,'track':TRACK,'owner':OWNER,'verification_level':MODE,'status':'RUNNING','verify_command':cmd,'python':sys.version,'packages':{p:importlib.metadata.version(p) for p in ['kickmix','numpy','scipy']},'submission_sha256':{p.name:sha(p) for p in SUB.iterdir() if p.is_file()},'verifier_sha256':{str(p.relative_to(ROOT)):sha(p) for p in (ROOT/'verifiers').rglob('*.py')}}
# Hash of the original .kmb/.zip; submission_sha256 retains extracted file hashes.
meta['upload_artifact']=dict(UPLOAD_ARTIFACT)
meta['dev_harness_sha256']=sha(ROOT/'src/common/dev_verify.py') if MODE=='dev_5min' else None
meta['timeout_seconds']=DEV_TIMEOUT_SECONDS if MODE=='dev_5min' else None
(OUT/'config.json').write_text(json.dumps({'track':TRACK,'experiment_id':EXPERIMENT_ID,'config_name':CONFIG_NAME,'mode':MODE,'command':cmd},indent=2))
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2))
env=dict(os.environ); env['PYTHONPATH']=str(ROOT/'verifiers'); env['OPENBLAS_NUM_THREADS']='1'; env['OMP_NUM_THREADS']='1'
print('Running:',cmd,flush=True)
with (OUT/'verify.stdout.txt').open('w') as stdout,(OUT/'verify.stderr.txt').open('w') as stderr:
    try:
        proc=subprocess.run(cmd,cwd=ROOT,env=env,stdout=stdout,stderr=stderr,
                            timeout=DEV_TIMEOUT_SECONDS if MODE=='dev_5min' else None)
        timed_out=False
    except subprocess.TimeoutExpired:
        proc=None
        timed_out=True
        meta['status']='TIMEOUT'
raw=(OUT/'verify.stdout.txt').read_text()
if timed_out: print('TIMEOUT: không có điểm hợp lệ; giảm mẫu trong profile rồi chạy lại. Không tính đây là lỗi mạch.')
try:
    result=json.loads('' if timed_out else raw)
    (OUT/'verifier.json').write_text(raw)
    meta['status']='PASS' if proc.returncode==0 and result.get('status')=='PASS' else 'FAIL'
    print(json.dumps(result,indent=2))
except ValueError:
    meta['status']='TIMEOUT' if timed_out else 'ERROR'; print(raw); print((OUT/'verify.stderr.txt').read_text()[-6000:])
meta['exit_code']=None if proc is None else proc.returncode; meta['finished_at_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2))
print('Saved:',OUT)


## Tải results về project
ZIP giữ cấu trúc `results/<bài>/<ID>/<config>/<chế độ>/<run-id>/`. Giải nén vào thư mục gốc project. Tên config tự lấy từ `.kmb` hoặc ZIP, không cần nhập và không cần upload JSON build. Ghi điểm full/dev_5min PASS vào cột tương ứng trong SCORES.md.


In [ ]:
archive=Path('/content')/f'GQA-results-{EXPERIMENT_ID}-{CONFIG_NAME}-{run_id}.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p,p.relative_to(ROOT))
files.download(str(archive))
